In [1]:
import pandas as pd
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score


In [2]:

CSV_PATH = "raw_wholesale_customers.csv"
OUT_PATH = "raw_wholesale_customers.csv"
FEATURES = ["Fresh", "Milk", "Grocery",
            "Frozen", "Detergents_Paper", "Delicassen"]
RANDOM_STATE = 42
K = 3


In [4]:

df = pd.read_csv(CSV_PATH)
# print(df.head())

X = df[FEATURES].copy()


In [ ]:
def iqr_fun(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return lower, upper


low_fresh,  high_fresh = iqr_fun(X["Fresh"])
low_milk,    high_milk = iqr_fun(X["Milk"])
low_grocery, high_grocery = iqr_fun(X["Grocery"])
low_frozen,  high_frozen = iqr_fun(X["Frozen"])
low_det,     high_det = iqr_fun(X["Detergents_Paper"])
low_deli,    high_deli = iqr_fun(X["Delicassen"])

X["Fresh"] = X["Fresh"].clip(lower=low_fresh,  upper=high_fresh)
X["Milk"] = X["Milk"].clip(lower=low_milk,    upper=high_milk)
X["Grocery"] = X["Grocery"].clip(lower=low_grocery, upper=high_grocery)
X["Frozen"] = X["Frozen"].clip(lower=low_frozen,  upper=high_frozen)
X["Detergents_Paper"] = X["Detergents_Paper"].clip(
    lower=low_det, upper=high_det)
X["Delicassen"] = X["Delicassen"].clip(lower=low_deli, upper=high_deli)


In [ ]:

df[FEATURES] = X

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
# print("\nScaled shape:", X_scaled.shape)
# print(X_scaled)

for k in range(1, 11):
    km = KMeans(n_clusters=k, n_init="auto", random_state=RANDOM_STATE)
    km.fit(X_scaled)
    # print(f"k={k} → SSE={km.inertia_:.2f}")


In [ ]:
kmeans = KMeans(n_clusters=K, n_init="auto", random_state=RANDOM_STATE)
labels = kmeans.fit_predict(X_scaled)

df["Cluster"] = labels.astype(int)
# print(df.head())   

print("\n=== K-MEAN ===")

sil = silhouette_score(X_scaled, labels)
dbi = davies_bouldin_score(X_scaled, labels)


# print(f"\n===== K = 3 =====")
print(f"Silhouette Score : {sil:.3f} (closer to +1 is better)")
print(f"Davies–Bouldin   : {dbi:.3f} (lower is better)")

sample_idx = [0, 1, 2]
sanity = df.loc[sample_idx, ["Channel", "Region"] + FEATURES + ["Cluster"]]

print("\n=== SANITY CHECK (3 Clients) ===")
print(sanity)



In [ ]:
#SECOND ALGORITHM

agg = AgglomerativeClustering(n_clusters=K)

agg_labels = agg.fit_predict(X_scaled)

df["Agglomerative_Cluster"] = agg_labels.astype(int)

agg_silhouette = silhouette_score(X_scaled, agg_labels)

print("\n=== Agglomerative Clustering ===")

# print(f"\n===== K = 3 =====")
print(f"Silhouette Score: {agg_silhouette:.4f}")

# Sanity Check (3 clients)
sample_idx = [0, 1, 2]

sanity = df.loc[
    sample_idx,
    ["Channel", "Region"] + FEATURES + ["Agglomerative_Cluster"]
]

print("\n=== AGGLOMERATIVE SANITY CHECK ===")

print(sanity)


In [ ]:
centers_scaled = kmeans.cluster_centers_
centers_original = scaler.inverse_transform(centers_scaled)

centers_df = pd.DataFrame(centers_original, columns=FEATURES)
centers_df.index.name = "Cluster"
print(centers_df.round(2))

In [ ]:
df.to_csv(OUT_PATH, index=False)
print(f"\nsegmented_wholesale_customers.csv → {OUT_PATH}")